# Tugas 2 | Crawling Berita

Kode ini dibuat untuk melakukan web crawling berita dari situs Detik.com pada kategori Olahraga dan Pendidikan. Proses crawling dimulai dari tanggal *31 Desember 2024* dan berjalan mundur hingga jumlah berita yang diambil mencapai *100 berita*. Dari setiap berita yang ditemukan, program mengambil **judul, link, kategori, tanggal, dan isi berita**. Seluruh hasil kemudian disimpan ke dalam file CSV sehingga dapat digunakan untuk analisis lebih lanjut.

berikut Code Crawling nya:

**Import Library**

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta

**Fungsi untuk crawling berita dari portal Detik**


Parameter:
- limit  : jumlah maksimum berita yang ingin diambil
- tahun  : tahun berita yang ingin dicrawl (default 2024)

Return:
- DataFrame berisi judul, link, kategori, tanggal, dan isi berita

In [ ]:
def crawl_detik(limit=100, tahun=2024):
    # URL dasar berdasarkan kategori
    base_urls = {
        "Olahraga": "https://sport.detik.com/indeks",
        "Pendidikan": "https://www.detik.com/edu/indeks"
    }

    hasil = []            # list untuk menyimpan hasil crawl
    total_berita = 0      # counter jumlah berita yang sudah diambil

    # header untuk menyamar sebagai browser (supaya tidak diblokir)
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/126.0.0.0 Safari/537.36"
    }

    # mulai dari tanggal 31 Desember tahun yang ditentukan (mundur ke belakang)
    tgl = datetime(tahun, 12, 31)

    # loop sampai jumlah berita terpenuhi atau sudah ganti tahun
    while total_berita < limit and tgl.year == tahun:
        # format tanggal untuk URL dan penyimpanan CSV
        tgl_str = tgl.strftime("%Y/%m/%d")    # format untuk URL
        tgl_csv = tgl.strftime("%Y-%m-%d")    # format untuk CSV

        # loop per kategori (Olahraga & Pendidikan)
        for kategori, base_url in base_urls.items():
            if total_berita >= limit:  # berhenti kalau sudah cukup
                break

            url = f"{base_url}?date={tgl_str}"
            print(f"🔎 Crawling {kategori} {tgl_str} -> {url}")

            try:
                # request halaman indeks berita berdasarkan kategori + tanggal
                r = requests.get(url, headers=headers, timeout=10)
                r.raise_for_status()
                soup = BeautifulSoup(r.text, "html.parser")

                # ambil semua artikel berita di halaman tersebut
                items = soup.find_all("article")
                if not items:
                    continue

                # proses setiap berita dalam daftar artikel
                for item in items:
                    if total_berita >= limit:
                        break

                    judul_tag = item.find("h3")
                    link_tag = item.find("a")

                    if judul_tag and link_tag:
                        # ambil judul & link berita
                        judul = judul_tag.get_text(strip=True)
                        link = link_tag["href"]

                        isi = ""  # isi artikel berita
                        try:
                            # buka link berita untuk ambil isi lengkap
                            r2 = requests.get(link, headers=headers, timeout=10)
                            r2.raise_for_status()
                            soup2 = BeautifulSoup(r2.text, "html.parser")

                            # ambil isi artikel dari div "detail__body-text"
                            content = soup2.find("div", class_="detail__body-text")
                            if content:
                                isi = " ".join([p.get_text(strip=True) for p in content.find_all("p")])
                        except Exception:
                            isi = "⚠️ Error ambil isi"

                        # simpan hasil ke list
                        hasil.append({
                            "Judul Berita": judul,
                            "Link": link,
                            "Kategori": kategori,
                            "Tanggal": tgl_csv,   # disimpan sebagai string
                            "Isi Berita": isi
                        })
                        total_berita += 1
                        print(f"✅ Disimpan ({total_berita}): {judul}")

            except Exception as e:
                # jika gagal request (misal timeout / error koneksi)
                print(f"⚠️ Gagal membuka {url}: {e}")

        # mundur ke tanggal sebelumnya
        tgl -= timedelta(days=1)

    # konversi hasil ke DataFrame
    df = pd.DataFrame(hasil)

    # pastikan kolom Tanggal tetap string (hindari auto-format Excel)
    df["Tanggal"] = df["Tanggal"].astype(str)
    return df



# Eksekusi utama
data = crawl_detik(limit=100, tahun=2024)

print("\nTotal berita berhasil dicrawl:", len(data))

# simpan hasil ke file CSV
data.to_csv("detik_berita_2024.csv", index=False, encoding="utf-8-sig")
print("✅ Hasil disimpan ke detik_berita_2024.csv")

🔎 Crawling Olahraga 2024/12/31 -> https://sport.detik.com/indeks?date=2024/12/31
✅ Disimpan (1): Marcus Gideon ke Ganda Pratama : Enggak Bisa Santai Kalau Mau Jago
✅ Disimpan (2): Megawati Tampil Gemilang, Red Sparks Gebuk IBK Altos 3-0
✅ Disimpan (3): Punya Tim & Semangat Baru, Jakarta Pertamina Enduro Siap Taklukkan Proliga
✅ Disimpan (4): Jakarta Pertamina Enduro Tak Mau Lagi Cuma Gaspol di Awal Proliga
✅ Disimpan (5): Ducati ke Honda: Bangun Motor Butuh Waktu, Tak Cuma Semalam
✅ Disimpan (6): Herry IP ke Malaysia: Tinggal Tunggu Draf Kontrak
✅ Disimpan (7): Tinggalkan Indonesia, Herry IP Pastikan Siap Latih Malaysia
🔎 Crawling Pendidikan 2024/12/31 -> https://www.detik.com/edu/indeks?date=2024/12/31
✅ Disimpan (8): 'UN' Sistem Baru Akan Diadakan Kemendikdasmen di 2026, Tulis Pendapatmu di POV!
✅ Disimpan (9): Penemuan Bangkai Paus Paling Langka di Dunia, Ilmuwan Ungkap Fakta Ini
✅ Disimpan (10): Kapan Perayaan Tahun Baru Pertama Kali Dirayakan? Sejak Ribuan Tahun Lalu!
✅ Disimpan (